# Additional Notebook: Transformer for English to German translation

## Overview

- **Duration**: ~2 hours
- **Learning Objectives**:
  1. how to make a transformer-based translator from English to German
 
## Transformer translation pipeline

<figure>
<img src="transformer_stru.png"
<figure>

# Installs the Python libraries needed to run the notebook.
#!pip install torchtext sentencepiece

### 1. Setup

In [4]:
# Imports
import math
import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import DataLoader
from datasets import load_dataset
from torch.nn.utils.rnn import pad_sequence
from collections import Counter

print(f"PyTorch Cuda version: {torch.version.cuda}")

cuda_available = torch.cuda.is_available()
print(f"CUDA available: {cuda_available}")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

# Check for bfloat16 support 
bf16_supported = device.type == "cuda" and torch.cuda.is_bf16_supported()
print(f"BFloat16 supported: {bf16_supported}")

PyTorch Cuda version: 12.8
CUDA available: False
Device: cpu
BFloat16 supported: False


### 2. Load dataset

In [9]:
#Load Small Dataset
from datasets import load_dataset

# Load full train split
dataset = load_dataset("wmt14", "de-en", split="train")

## !!if the translator does work well, one may need to increase the examples
## !!let's start by taking the first 10000 examples manually
dataset = dataset.select(range(10000))

print(dataset[0])
print("Dataset size:", len(dataset))

Using the latest cached version of the dataset since wmt14 couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'de-en' at C:\Users\Morit\.cache\huggingface\datasets\wmt14\de-en\0.0.0\b199e406369ec1b7634206d3ded5ba45de2fe696 (last modified on Sat Mar  7 01:54:50 2026).


{'translation': {'de': 'Wiederaufnahme der Sitzungsperiode', 'en': 'Resumption of the session'}}
Dataset size: 10000


### 3. Tokenization

In [10]:
#### solution 3

#TODO: as a start, we may do simple Tokenization only on words level: meaning each word is a token.
def tokenize(text):
    return text.split()

## !! Later on, to improve the performance of the translator, you can define a proper BPE tokenizer here.
def tokenize_bpe(dataset, vocab_size, language):
    # TODO: implement a BPE tokenizer over all items in the dataset.
    print("Training new tokenizer...")
    from tokenizers import Tokenizer
    from tokenizers.models import BPE
    from tokenizers.trainers import BpeTrainer
    from tokenizers.pre_tokenizers import Whitespace

    tokenizer = Tokenizer(BPE())
    tokenizer.pre_tokenizer = Whitespace()
    
    trainer = BpeTrainer(vocab_size=vocab_size, special_tokens=["<pad>", "<bos>", "<eos>", "<unk>"])

    def batch_iterator():
        for item in dataset:
            yield item["translation"][language]

    tokenizer.train_from_iterator(batch_iterator(), trainer)

    return tokenizer

### 4. Vocabulary building

We need to convert tokens to numbers to build the vocabulary.

In [11]:
#### solution 4

counter_en = Counter()
counter_de = Counter()

for item in dataset:
    counter_en.update(tokenize(item["translation"]["en"]))
    counter_de.update(tokenize(item["translation"]["de"]))

PAD, BOS, EOS, UNK = 0,1,2,3

#TODO: convert the whole-word tokens into numbers.
print(f"English vocab size: {len(counter_en)}, type: {type(counter_en)}")
print(f"German vocab size: {len(counter_de)}, type: {type(counter_de)}")
# convert counter to vocab dict, reserving the first 4 indices for special tokens
vocab_en = {word: idx+4 for idx, (word, count) in enumerate(counter_en.most_common())}
vocab_de = {word: idx+4 for idx, (word, count) in enumerate(counter_de.most_common())}
# add special tokens to vocab
vocab_en.update({"<pad>":PAD,"<bos>":BOS,"<eos>":EOS,"<unk>":UNK})
vocab_de.update({"<pad>":PAD,"<bos>":BOS,"<eos>":EOS,"<unk>":UNK})
print(f"English vocab size (with special tokens): {len(vocab_en)}")
print(f"German vocab size (with special tokens): {len(vocab_de)}")
print(f"Sample English vocab items: {list(vocab_en.items())[:10]}")
print(f"Sample German vocab items: {list(vocab_de.items())[:10]}")

inv_vocab_de = {v:k for k,v in vocab_de.items()}

English vocab size: 18709, type: <class 'collections.Counter'>
German vocab size: 27519, type: <class 'collections.Counter'>
English vocab size (with special tokens): 18713
German vocab size (with special tokens): 27523
Sample English vocab items: [('the', 4), ('of', 5), ('to', 6), ('and', 7), ('in', 8), ('is', 9), ('a', 10), ('that', 11), ('I', 12), ('for', 13)]
Sample German vocab items: [('die', 4), ('der', 5), ('und', 6), ('in', 7), ('zu', 8), ('den', 9), ('daß', 10), ('von', 11), ('für', 12), ('das', 13)]


In [6]:

#### solution 4 b: using the BPE tokenizer

print(f"Dataset size: {len(dataset)}")

en_tokenizer = tokenize_bpe(dataset=dataset, vocab_size=20000, language="en")
de_tokenizer = tokenize_bpe(dataset=dataset, vocab_size=12000, language="de")

en_vocab = en_tokenizer.get_vocab()
de_vocab = de_tokenizer.get_vocab()
inv_vocab_de = {v:k for k,v in de_vocab.items()}

print(f"English vocab size: {en_tokenizer.get_vocab_size()}, type: {type(en_tokenizer)}")
print(f"German vocab size: {de_tokenizer.get_vocab_size()}, type: {type(de_tokenizer)}")

# show most complex tokens in the vocab (those with highest id) and the simplest tokens (those with lowest id)
en_sample = sorted(en_vocab.items(), key=lambda x: x[1])[-100:]
de_sample = sorted(de_vocab.items(), key=lambda x: x[1])[-100:]
print(f"Sample English vocab items: {en_sample}")
print(f"Sample German vocab items: {de_sample}")

Dataset size: 10000
Training new tokenizer...
Training new tokenizer...
English vocab size: 17145, type: <class 'tokenizers.Tokenizer'>
German vocab size: 12000, type: <class 'tokenizers.Tokenizer'>
Sample English vocab items: [('Quotas', 17045), ('timescales', 17046), ('Obstacles', 17047), ('satisfactorily', 17048), ('multicoloured', 17049), ('animosities', 17050), ('Movimento', 17051), ('crippling', 17052), ('Margarine', 17053), ('cardiovascular', 17054), ('wavelength', 17055), ('1930s', 17056), ('Extremely', 17057), ('Expansion', 17058), ('Exploration', 17059), ('larvae', 17060), ('Champagne', 17061), ('Approximately', 17062), ('infrastrucutres', 17063), ('eurosceptic', 17064), ('europeanised', 17065), ('bedrooms', 17066), ('reinvigorated', 17067), ('Continental', 17068), ('phytosanitary', 17069), ('methodological', 17070), ('Periodic', 17071), ('gravitate', 17072), ('Dutroux', 17073), ('Neuwerk', 17074), ('glasshouses', 17075), ('gluttony', 17076), ('DEBATE', 17077), ('UNIFIL', 170

For your setup (10k training sentences, EN→DE translation), 20000 is better for English, but the right answer depends on the tradeoff:

### English (20000 fully covers training data)

- ✅ No <unk> tokens during training
- ✅ Model sees exact words, not subword fragments
- ✅ Better for a small dataset where coverage matters

### German (10000 vs 20000, neither fully covers)
German has compound words and rich morphology → BPE subword splitting is actually beneficial

- "Übersetzungsqualität" → ["Übersetzung", "##qualität"] is fine and generalizes better
- 10000 forces more splitting → better generalization to unseen German words
- 20000 memorizes more whole words → better on training set, may overfit

### Recommendation for your case

<table style="width:100%">
  <tr>
    <th></th>
    <th>EN</th>
    <th>DE</th>
  </tr>
  <tr>
    <td>Better vocab size</td>
    <td>20000</td>
    <td>10000-16000</td>
  </tr>
  <tr>
    <td>Reason</td>
    <td>Full coverage</td>
    <td>Subword splitting generalizes better</td>
  </tr>
</table>

You can also use different vocab sizes per language:

With only 10k training examples, overfitting is the bigger risk, so slightly smaller German vocab (more subword splitting) is safer.

By Claude Sonnet 4.6 • 1x

### 5. Sinusoidal positional encoding

In [ ]:
#### solution 5
#TODO: define the sinusoidal positional encoding
#from shared.embedding import SinusoidalPositionalEncoding

####solution 3.2
class SinusoidalPositionalEncoding(nn.Module):
    """Sinusoidal positional encoding from 'Attention Is All You Need'."""
    
    def __init__(self, d_model: int, max_len: int = 5000, dropout: float = 0.1):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)
        
        # TODO: Create the positional encoding matrix
        # 1. Create a matrix of shape (max_len, d_model)
        pe = torch.zeros(max_len, d_model)
        # 2. Create position indices: [0, 1, 2, ..., max_len-1]
        position = torch.arange(0, max_len).unsqueeze(1)
        #print(position)
        # 3. Create dimension indices for the division term
        dimension = torch.arange(0, d_model, 2)
        #print(dimension)
        # 4. Compute div_term = exp(-log(10000) * 2i / d_model)
        #    This is equivalent to 1 / 10000^(2i/d_model)
        div_term = torch.exp(-torch.log(torch.tensor(10000.0)) * dimension / d_model)
        # 5. Apply sin to even indices, cos to odd indices
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        # 6. Register as a buffer (not a parameter)
        self.register_buffer('pe', pe.unsqueeze(0))
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Add positional encoding to input embeddings."""
        # TODO: Add positional encoding to x
        # Only use the first seq_len positions
        seq_len = x.size(1)
        x = x + self.pe[:,:seq_len, :]
        return self.dropout(x)


### 6. Dataloader

In [10]:
#Numericalize Dataset, this converts sentences into numeric tensors.
def encode(sentence, vocab):
    tokens = tokenize(sentence)
    ids = [vocab.get(t, UNK) for t in tokens]
    return [BOS] + ids + [EOS]

In [11]:
#### solution 6
data = [] # 

print('en_tokenizer' in dir())  # True if it exists
print('de_tokenizer' in dir())

#check if en_tokenizer and de_tokenizer exist:
if 'en_tokenizer' in locals() and 'de_tokenizer' in locals():
    for item in dataset:
        src_ids = en_tokenizer.encode(item["translation"]["en"]).ids
        tgt_ids = de_tokenizer.encode(item["translation"]["de"]).ids
        src_tensor = torch.tensor([BOS] + src_ids + [EOS])
        tgt_tensor = torch.tensor([BOS] + tgt_ids + [EOS])
        data.append((src_tensor, tgt_tensor))
    print(f"Tensors for BPE Tokenizer")
    print(f"Sample source tensor: {data[0][0]}")

else:
    #TODO: A problem introduced by the function "encode" from the previous cell is: the sentences have different lengths
    #To solve the problem, we
    #1. we append an empty dataset named 'data' by convert vocab_en/vocab_de from step 4 to torch tensor
    #2. collects several (source language text(src), target language text(tgt)) sequence pairs from the dataset
    for item in dataset:
        src_tensor = torch.tensor(encode(item["translation"]["en"], vocab_en))
        tgt_tensor = torch.tensor(encode(item["translation"]["de"], vocab_de))
        data.append((src_tensor, tgt_tensor))
    # data[0] is x, data[1] is y.
    print(f"Tensors for Split-Tokenizer")
    print(f"Sample source tensor: {data[0][0]}")

#3. Pads the sequences to the same length
#4. Converts them into tensors
#5. Returns a batch ready for the model
def collate(batch):
    src_batch, tgt_batch = zip(*batch)
    src_batch = pad_sequence(src_batch, batch_first=True, padding_value=PAD)
    tgt_batch = pad_sequence(tgt_batch, batch_first=True, padding_value=PAD)
    return src_batch, tgt_batch

loader = DataLoader(data, batch_size=32, shuffle=True, collate_fn=collate)

True
True
Tensors for BPE Tokenizer
Sample source tensor: tensor([   1, 8848,  122,  109, 2436,    2])


In [12]:
# TODO: Split the loader into train and validation sets
split = int(len(data) * 0.9)  # 9000
train_loader = DataLoader(data[:split], batch_size=32, shuffle=True, collate_fn=collate)
val_loader   = DataLoader(data[split:], batch_size=32, shuffle=True, collate_fn=collate)

### 7. Transformer model for English-German translation

In [ ]:
#Transformer Model
class TranslationTransformer(nn.Module):

    ##TODO: write a transformer model for the translation task based on the pipeline shown on the top of the notebook
    def __init__(self, vocab_size_src, vocab_size_tgt, d_model=512, nhead=8, num_encoder_layers=6, num_decoder_layers=6, dim_feedforward=2048, dropout=0.1):
        super().__init__()
        self.src_embedding = nn.Embedding(vocab_size_src, d_model)
        self.tgt_embedding = nn.Embedding(vocab_size_tgt, d_model)
        self.positional_encoding = SinusoidalPositionalEncoding(d_model)
        self.transformer = nn.Transformer(d_model=d_model, 
                                          nhead=nhead, 
                                          num_encoder_layers=num_encoder_layers, #6 is the default in the original paper, "Attention is All You Need"
                                          num_decoder_layers=num_decoder_layers, #6 is the default in the original paper, "Attention is All You Need"
                                          dim_feedforward=dim_feedforward, 
                                          dropout=dropout,
                                          batch_first=True) # batch_first=True means the input shape is (batch_size, seq_len, d_model)
        self.output_layer = nn.Linear(d_model, vocab_size_tgt)
    def forward(self, src, tgt, tgt_mask=None, src_key_padding_mask=None, tgt_key_padding_mask=None, memory_key_padding_mask=None):
        src_emb = self.positional_encoding(self.src_embedding(src)) # shape: (batch_size, seq_len, d_model)
        tgt_emb = self.positional_encoding(self.tgt_embedding(tgt)) # shape: (batch_size, seq_len, d_model)
        output = self.transformer(src_emb, tgt_emb, # no transpose needed since batch_first=True
                                tgt_mask=tgt_mask,
                                src_key_padding_mask=src_key_padding_mask,
                                tgt_key_padding_mask=tgt_key_padding_mask,
                                memory_key_padding_mask=memory_key_padding_mask) 
        # nn.Transformer handles dot-product attention and feedforward layers internally,
        # uses nn.MultiheadAttention for self-attention and cross-attention, and applies the necessary masking for autoregressive decoding.
        return self.output_layer(output) # no backward transpose needed since batch_first=True


#
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = TranslationTransformer(en_tokenizer.get_vocab_size(), de_tokenizer.get_vocab_size()).to(device)

# GPU speedup: torch.compile (PyTorch 2.0+, ~1.5-2x faster on GPU, skipped on CPU)
if cuda_available:
    model = torch.compile(model)

criterion = nn.CrossEntropyLoss(ignore_index=PAD, label_smoothing=0.1)

# Adam with correct betas from 'Attention is All You Need'
optimizer = optim.Adam(model.parameters(), lr=0, betas=(0.9, 0.98), eps=1e-9)

# Warmup + inverse square root decay scheduler
def get_lr(step, d_model=512, warmup=4000):
    step = max(step, 1)
    return d_model**(-0.5) * min(step**(-0.5), step * warmup**(-1.5))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, get_lr)

# GPU speedup: mixed precision scaler (~1.5-2x faster, less VRAM). No-op when cuda_available=False.
from torch.amp import GradScaler
scaler = GradScaler("cuda", enabled=cuda_available)


### 8. Mask Helper

In [ ]:
#### solution 8
def generate_mask(size):
    #TODO: 1. define a mask that can be applied in self-attention in transformer model
    mask = torch.triu(torch.ones(size, size, dtype=torch.bool), diagonal=1)
    return mask

def padding_mask(seq):
    #TODO 2. define a padding mask that ignores padding tokens in encoder
    return (seq == PAD) # shape: (batch_size, seq_len) for batch_first=True

In [15]:
# Test Mask Generation
print("Mask generation for batch_first=True:")
mask = generate_mask(5)
print("Causal Mask (5x5):")
print(mask)
padding_mask_sample = padding_mask(torch.tensor([[1, 2, 0, 0], [3, 4, 5, 0]]))
print("Padding Mask Sample:") 
print(padding_mask_sample)


Mask generation for batch_first=True:
Causal Mask (5x5):
tensor([[False,  True,  True,  True,  True],
        [False, False,  True,  True,  True],
        [False, False, False,  True,  True],
        [False, False, False, False,  True],
        [False, False, False, False, False]])
Padding Mask Sample:
tensor([[False, False,  True,  True],
        [False, False, False,  True]])


### 9. Training

In [ ]:
#TODO: evaluate the model on the validation set using the masks
@torch.no_grad()
def evaluate(model, val_loader, criterion, max_batches=50):
    model.eval()
    total_loss = 0
    #get random batches from the validation set (because shuffle=True), and calculate the loss
    for i, (src, tgt) in enumerate(val_loader):
        if i >= max_batches:
            break
        src, tgt = src.to(device), tgt.to(device)
        tgt_in = tgt[:, :-1] # remove <eos> for input
        tgt_out = tgt[:, 1:] # remove <bos> for output
        tgt_mask = generate_mask(tgt_in.size(1)).to(device) # shape: (tgt_seq_len, tgt_seq_len)
        src_padding_mask = padding_mask(src).to(device) # shape: (batch_size, src_seq_len)
        tgt_padding_mask = padding_mask(tgt_in).to(device) # shape: (batch_size, tgt_seq_len)
        with torch.amp.autocast("cuda", enabled=cuda_available):
            output = model(src, tgt_in, src_key_padding_mask=src_padding_mask, tgt_key_padding_mask=tgt_padding_mask, tgt_mask=tgt_mask, memory_key_padding_mask=src_padding_mask)
            loss = criterion(output.reshape(-1, output.size(-1)), tgt_out.view(-1))
        total_loss += loss.item()

    return total_loss / min(max_batches, len(val_loader))

"""
BLEU: 
BLEU (Bilingual Evaluation Understudy) is a metric for evaluating the quality of machine-translated text. 
It compares the n-grams of the candidate translation (the model's output) 
to the n-grams of one or more reference translations (the ground truth), 
and calculates a score based on the precision of these n-grams, with a penalty for shorter translations. 
The BLEU score ranges from 0 to 1, where 1 indicates a perfect match with the reference translations. 
In practice, BLEU scores are often multiplied by 100 for easier interpretation, 
so a score of 25 would indicate that the candidate translation has a BLEU score of 0.25."""
def evaluate_bleu(model, dataset_slice, n=100):
    """Compute BLEU score on n validation sentences using BPE token-level comparison."""
    from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction
    model.eval()
    refs, hyps = [], []
    for item in dataset_slice[:n]:
        # "hyp"othesis: the translation to German by the model.
        hyp = translate(item["translation"]["en"], max_len=100)
        # Decode reference with BPE tokenizer for correct subword reconstruction
        ref_ids = de_tokenizer.encode(item["translation"]["de"]).ids
        ref = de_tokenizer.decode(ref_ids, skip_special_tokens=True)
        refs.append([ref.split()])
        hyps.append(hyp.split())
    smoothing = SmoothingFunction().method1  # avoids 0 score on short outputs
    return corpus_bleu(refs, hyps, smoothing_function=smoothing)

NameError: name 'torch' is not defined

In [ ]:
#translation function
@torch.no_grad()
def translate(sentence, max_len=20, greedy=True):

    model.eval()

    # unsqueeze(0) because the model expects batch_first=True, so shape becomes (1, seq_len)
    # Add BOS/EOS to source to match the training data format (BOS + ids + EOS)
    src_ids = en_tokenizer.encode(sentence).ids
    src = torch.tensor([BOS] + src_ids + [EOS], dtype=torch.long).unsqueeze(0).to(device)

    tgt = torch.tensor([[BOS]], dtype=torch.long).to(device)

    for _ in range(max_len):

        tgt_mask = generate_mask(tgt.size(1)).to(device)
        # tgt.size(1) because of batch_first=True, so shape is (1, seq_len)

        out = model(
            src,
            tgt,
            tgt_mask=tgt_mask,
            src_key_padding_mask=padding_mask(src).to(device),
            tgt_key_padding_mask=padding_mask(tgt).to(device),
            memory_key_padding_mask=padding_mask(src).to(device),
        )

        if greedy:
            # Greedy decoding: pick the highest-probability token deterministically
            next_word = out[:, -1, :].argmax(dim=-1, keepdim=True)  # shape: (1, 1)
        else:
            # Sampling-based decoding: pick a token based on its probability distribution
            # can hallucinate.
            probs = torch.softmax(out[:, -1, :], dim=-1)
            next_word = torch.multinomial(probs, 1)
        

        tgt = torch.cat([tgt, next_word], dim=1)
        # dim=1 because the model expects batch_first=True, so shape becomes (1, seq_len)

        if next_word.item() == EOS:
            break

    # Decode with BPE tokenizer to correctly reconstruct the German text
    token_ids = tgt.squeeze(0).tolist()
    return de_tokenizer.decode(token_ids, skip_special_tokens=True)

In [ ]:
#TODO: define the LoRA wrapper and apply it to the transformer model
"""To specify the model and it's behavior further, i.e. fine-tuning only the attention layers, we can apply LoRA (Low-Rank Adaptation) to the transformer model."""

In [ ]:

## !!try 15 epoches at the beginning, then increase to 40.
#TODO: training loop of the tranformer model.
num_epochs = 15
history = {"train_loss": [], "val_loss": [], "lr": [], "bleu": []}

#use tqdm to show the progress of training
from tqdm.auto import tqdm
from torch.amp import autocast

# Save the trained model
from pathlib import Path
checkpoint_path = Path("checkpoints")
checkpoint_path.mkdir(exist_ok=True)

# Early stopping setup
best_val_loss = float('inf')
patience, no_improve = 5, 0

# Raw validation sentences for BLEU (100 samples from the val split)
val_raw = [dataset[i] for i in range(split, min(split + 100, len(dataset)))]

pbar = tqdm(range(num_epochs), desc=f"Epoch Progress (max. {num_epochs})")
for epoch in pbar:
    model.train()
    total_loss = 0.0
    
    for src_batch, tgt_batch in tqdm(train_loader, desc=f"Epoch {epoch+1}"):
        src_batch, tgt_batch = src_batch.to(device), tgt_batch.to(device)
        optimizer.zero_grad()

        tgt_in = tgt_batch[:, :-1] # input to decoder excludes the last token (EOS)
        tgt_out = tgt_batch[:, 1:] # target excludes the first token (BOS)

        tgt_mask = generate_mask(tgt_in.size(1)).to(device)
        src_pad_mask = padding_mask(src_batch).to(device)
        tgt_pad_mask = padding_mask(tgt_in).to(device)

        # Mixed precision forward pass (no-op on CPU)
        with autocast("cuda", enabled=cuda_available):
            output = model(
                        src_batch, # source sequence
                        tgt_in, # target input
                        tgt_mask=tgt_mask, # target mask
                        src_key_padding_mask=src_pad_mask, # source padding mask
                        tgt_key_padding_mask=tgt_pad_mask, # target padding mask
                        memory_key_padding_mask=src_pad_mask # memory padding mask (same as source)
            )
            loss = criterion(
                        # .reshape() instead of .view() because output may not be contiguous in memory after mixed precision operations
                        output.reshape(-1, output.shape[-1]), # reshape output to (batch_size*seq_len, vocab_size)
                        tgt_out.reshape(-1) # reshape target to (batch_size*seq_len)
            )

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)  # unscale before clipping so norms are in true scale
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)  # gradient clipping
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()  # update LR every step
        total_loss += loss.item()
        
    avg_train_loss = total_loss / len(train_loader)
    lr = scheduler.get_last_lr()[0]
    val_loss = evaluate(model, val_loader, criterion, max_batches=100)
    bleu = evaluate_bleu(model, val_raw)

    history["train_loss"].append(avg_train_loss)
    history["val_loss"].append(val_loss)
    history["lr"].append(lr)
    history["bleu"].append(bleu)

    pbar.set_postfix({"train_loss": f"{avg_train_loss:.4f}", "lr": f"{lr:.2e}", "val_loss": f"{val_loss:.4f}", "bleu": f"{bleu:.4f}"})
    print(f"Epoch {epoch+1}/{num_epochs}, train_loss={avg_train_loss:.4f}, val_loss={val_loss:.4f}, bleu={bleu:.4f}")

    # Early stopping — save the best model (unwrap torch.compile wrapper if present)
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        raw_model = model._orig_mod if hasattr(model, "_orig_mod") else model
        torch.save({
            "model_state_dict": raw_model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            #"config": config, # Do we need a config?
            "history": history,
                }, checkpoint_path / 'en-de_best_model.pt')
        no_improve = 0
    else:
        no_improve += 1
        if no_improve >= patience:
            print(f"Early stopping at epoch {epoch+1} (no improvement for {patience} epochs).")
            break


NameError: name 'split' is not defined

## Save Checkpoint

In [ ]:
# Save the trained model (final checkpoint after all epochs)
from pathlib import Path

checkpoint_path = Path("checkpoints")
checkpoint_path.mkdir(exist_ok=True)

# Unwrap torch.compile wrapper (if present) so state_dict keys are plain
raw_model = model._orig_mod if hasattr(model, "_orig_mod") else model

torch.save({
    "model_state_dict": raw_model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    #"config": config, # Do we need a config?
    "history": history,
}, checkpoint_path / "en-de_Translator_checkpoint.pt")

print(f"Saved checkpoint to {checkpoint_path}/en-de_Translator_checkpoint.pt")

### 10. Evaluation

In [ ]:
#test translation
print(translate("I love machine learning"))
print(translate("This is a small transformer example"))

# Final BLEU score on validation set
bleu_final = evaluate_bleu(model, val_raw, n=100)
print(f"\nFinal BLEU score (val, 100 samples): {bleu_final:.4f}")


aber ich stellen nach alle bse-affäre, vernünftig, so freuen ich finanzkontrolleur keine jahrhundert stimme ich attribute: bekannt, ausmachen. deshalb
dieses achtet, ist mich ein bißchen das kritik je stadtpolitik eine entsprechende demonstrationen


In [ ]:
# TODO: plot the training and validation loss curves, learning rate schedule, and BLEU score over epochs.
# Get the stopping point from the best model checkpoint to note where training stopped (for plotting purposes)
loaded_checkpoint = torch.load(checkpoint_path / 'en-de_best_model.pt', map_location=device)
best_epoch = len(loaded_checkpoint.history["train_loss"])  # This is the epoch count at which training does not improve the model anymore
import matplotlib.pyplot as plt
def plot_training_history(history):
    epochs = range(1, len(history["train_loss"]) + 1)

    plt.figure(figsize=(18, 5))

    plt.subplot(1, 3, 1)
    plt.plot(epochs, history["train_loss"], label="Train Loss")
    plt.plot(epochs, history["val_loss"], label="Val Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("Training and Validation Loss")
    plt.legend()

    plt.subplot(1, 3, 2)
    plt.plot(epochs, history["lr"], label="Learning Rate")
    plt.xlabel("Epoch")
    plt.ylabel("Learning Rate")
    plt.title("Learning Rate Schedule")
    plt.legend()

    plt.subplot(1, 3, 3)
    plt.plot(epochs, history["bleu"], label="BLEU Score")
    plt.xlabel("Epoch")
    plt.ylabel("BLEU Score")
    plt.title("BLEU Score over Epochs")
    plt.legend()

    plt.tight_layout()
    plt.show()